# 00 — Baseline: NYC green taxi trip duration

Minimum viable model. Target: trip duration in minutes.
Features: `PU_DO` (pickup–dropoff zone pair) and `trip_distance`.
Model: `DictVectorizer` + `LinearRegression`.

Run from the `notebooks/` folder. Paths are resolved relative to the repo root.

In [1]:
from pathlib import Path
import pickle
import urllib.request

import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import make_pipeline

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR, MODELS_DIR, REPORTS_DIR = ROOT / "data", ROOT / "models", ROOT / "reports"
for d in (DATA_DIR, MODELS_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

MONTH = "2024-01"
FILE = DATA_DIR / f"green_tripdata_{MONTH}.parquet"
URL = f"https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_{MONTH}.parquet"
SEED = 42

## 1. Download one month of data

In [2]:
if not FILE.exists():
    print(f"Downloading {URL}")
    urllib.request.urlretrieve(URL, FILE)
print(FILE, f"{FILE.stat().st_size / 1e6:.1f} MB")

/home/mohammed_hany/projects/mlops-practitioner/data/green_tripdata_2024-01.parquet 1.4 MB


## 2. Load and compute the target

In [3]:
df = pd.read_parquet(FILE)
print(df.shape)

df["duration"] = (
    df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
).dt.total_seconds() / 60

df["duration"].describe(percentiles=[0.01, 0.5, 0.95, 0.99])

(56551, 20)


count    56551.000000
mean        18.171379
std         75.464751
min          0.000000
1%           0.066667
50%         11.400000
95%         33.683333
99%         59.825000
max       1438.983333
Name: duration, dtype: float64

## 3. Clean and engineer features

Keep trips between 1 and 60 minutes (drops zero-length records and meters left running).
`PU_DO` is the pickup and dropoff zone IDs joined as one categorical string, so each route gets its own weight.

In [4]:
n_before = len(df)
df = df[(df["duration"] >= 1) & (df["duration"] <= 60)].copy()
print(f"Kept {len(df):,} of {n_before:,} trips ({len(df) / n_before:.1%})")

df["PU_DO"] = df["PULocationID"].astype(str) + "_" + df["DOLocationID"].astype(str)
df["trip_distance"] = df["trip_distance"].astype(float)

CATEGORICAL, NUMERICAL, TARGET = ["PU_DO"], ["trip_distance"], "duration"

Kept 54,373 of 56,551 trips (96.1%)


## 4. Train / validation split

Time-based: the last 7 days of the month are validation. This mimics production, where the model predicts trips it has never seen, better than a random shuffle.

In [5]:
cutoff = df["lpep_pickup_datetime"].max().normalize() - pd.Timedelta(days=6)
train_df = df[df["lpep_pickup_datetime"] < cutoff]
val_df = df[df["lpep_pickup_datetime"] >= cutoff]
print(f"Cutoff {cutoff.date()}  train={len(train_df):,}  val={len(val_df):,}")

def to_dicts(frame):
    return frame[CATEGORICAL + NUMERICAL].to_dict(orient="records")

X_train, y_train = to_dicts(train_df), train_df[TARGET].values
X_val, y_val = to_dicts(val_df), val_df[TARGET].values

Cutoff 2024-01-25  train=41,692  val=12,681


## 5. Fit the baseline

In [6]:
model = make_pipeline(DictVectorizer(), LinearRegression())
model.fit(X_train, y_train)
print("Features:", len(model[0].feature_names_))

Features: 4400


## 6. Evaluate

In [7]:
y_pred = model.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
mae = mean_absolute_error(y_val, y_pred)

# Reference point: always predicting the training mean
naive = np.full_like(y_val, y_train.mean())
naive_rmse = root_mean_squared_error(y_val, naive)

print(f"Validation RMSE: {rmse:.3f} min")
print(f"Validation MAE:  {mae:.3f} min")
print(f"Naive mean RMSE: {naive_rmse:.3f} min")

Validation RMSE: 6.118 min
Validation MAE:  3.945 min
Naive mean RMSE: 9.342 min


## 7. Save the model

In [8]:
model_path = MODELS_DIR / "baseline.pkl"
with open(model_path, "wb") as f:
    pickle.dump(model, f)
print("Saved", model_path)

# Sanity check: reload and predict one trip
with open(model_path, "rb") as f:
    reloaded = pickle.load(f)
print(reloaded.predict([{"PU_DO": "74_75", "trip_distance": 1.5}]))

Saved /home/mohammed_hany/projects/mlops-practitioner/models/baseline.pkl
[9.09700033]


## 8. Write metrics to the top of `reports/module-1.md`

Replaces only the metrics block between the markers, so anything else you write in the report is kept.

In [9]:
report = REPORTS_DIR / "module-1.md"
START, END = "<!-- baseline-metrics:start -->", "<!-- baseline-metrics:end -->"
block = f"""{START}
## Baseline metrics

| Metric | Value (minutes) |
|---|---|
| Validation RMSE | {rmse:.3f} |
| Validation MAE | {mae:.3f} |

Data: green taxi {MONTH}, time split at {cutoff.date()} (train {len(train_df):,} / val {len(val_df):,} trips).
Model: DictVectorizer + LinearRegression on `PU_DO`, `trip_distance`. Naive mean RMSE: {naive_rmse:.3f}.
{END}"""

existing = report.read_text() if report.exists() else "# Module 1 report\n"
if START in existing:
    rest = existing.split(END, 1)[1].lstrip("\n")
else:
    rest = existing
report.write_text(block + "\n\n" + rest)
print(report.read_text())

<!-- baseline-metrics:start -->
## Baseline metrics

| Metric | Value (minutes) |
|---|---|
| Validation RMSE | 6.118 |
| Validation MAE | 3.945 |

Data: green taxi 2024-01, time split at 2024-01-25 (train 41,692 / val 12,681 trips).
Model: DictVectorizer + LinearRegression on `PU_DO`, `trip_distance`. Naive mean RMSE: 9.342.
<!-- baseline-metrics:end -->

# Module 1 report

